In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import polars as pl
import torch
from sentence_transformers import CrossEncoder

# --- CONFIG ---
batchSize = 256     # Batch size for GPU processing
parquet_path = "../../data/raw/item_correlations.parquet"
output_path = "../../data/processed/item_correlations_with_cross.parquet"

# Create output folder if it does not yet exist
os.makedirs("../../data/processed", exist_ok=True)

# --- Load Data ---
dat = pl.read_parquet(parquet_path)
# Convert parameter pairs into a list format for the CrossEncoder
pairs = dat.select(["Parameter1", "Parameter2"]).to_numpy().tolist()

# --- Define the Models ---
# Models for sentiment, logical entailment, and semantic similarity
models_to_run = [
    {"name": "cardiffnlp/twitter-roberta-base-sentiment-latest", "nick": "sentiment"},
    {"name": "cross-encoder/nli-deberta-v3-base", "nick": "nli"},
    {"name": "mixedbread-ai/mxbai-rerank-large-v1", "nick": "similarity"}
]

# --- Execution Loop ---
for m in models_to_run:
    print(f"\n--- Loading {m['name']} ---")
    # Initialize CrossEncoder on CUDA with half-precision
    model = CrossEncoder(m['name'], device='cuda', model_kwargs={"torch_dtype": torch.float16})
    
    # Process multi-class models (NLI and Sentiment)
    if m['nick'] in ["nli", "sentiment"]:
        raw_scores = model.predict(pairs, batch_size=batchSize, show_progress_bar=True, apply_softmax=True)
        
        if m['nick'] == "nli":
            dat = dat.with_columns([
                pl.Series("contradiction", raw_scores[:, 0]),
                pl.Series("entail", raw_scores[:, 1]),
                pl.Series("logic_neutral", raw_scores[:, 2])
            ])
        # Unpack Sentiment specific probability vectors
        else: 
            dat = dat.with_columns([
                pl.Series("pair_negative", raw_scores[:, 0]),
                pl.Series("pair_positive", raw_scores[:, 2])
            ])
    # Process single-score similarity models
    else:
        scores = model.predict(pairs, batch_size=batchSize, show_progress_bar=True)
        dat = dat.with_columns(pl.Series(m['nick'], scores))
    
    # Cleanup GPU memory between model swaps
    del model
    torch.cuda.empty_cache()

# --- Feature Engineering ---
dat = dat.with_columns([
    # Interaction between reranker and NLI entailment
    (pl.col("similarity") * pl.col("entail")).alias("thematic_intensity"),

    # Similarity score weighted by contradiction probability
    (pl.col("similarity") * pl.col("contradiction")).alias("logical_friction"),

    # Calculate net sentiment polarity
    (pl.col("pair_positive") - pl.col("pair_negative")).alias("sentiment_balance")
])

# --- Export Data ---
# Save the engineered feature set to disk
dat.write_parquet(output_path)
print(f"Done! Saved to {output_path}")

c:\Users\alexm\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



--- Loading cardiffnlp/twitter-roberta-base-sentiment-latest ---


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 3414.30it/s]


RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Batches: 100%|█████████▉| 484/486 [04:01<00:00,  3.99it/s]

Batches: 100%|█████████▉| 485/486 [04:01<00:00,  4.00it/s]

Batches: 100%|██████████| 486/486 [04:01<00:00,  4.00it/s]

Batches: 100%|██████████| 486/486 [04:01<00:00,  2.01it/s]

Done! Saved to ../../data/processed/item_correlations_with_cross.parquet
